# 02 — Log Parsing & Session Creation
## HDFS Log Dataset (ไฟล์เต็ม 11,175,629 บรรทัด)

**เป้าหมาย:** แปลง log ดิบ → Event ID ด้วย Drain3 → จัดกลุ่มเป็น session ตาม `block_id` → ผูก label → แบ่ง train / val / test

**Analogy:** เหมือน SOC analyst ที่ต้องรวม log หลายพันบรรทัดของ incident เดียวกันมาเรียงเป็น timeline ก่อน แล้วถึงจะตัดสินได้ว่า incident นั้นเป็นของจริงหรือไม่ ถ้าดูแค่บรรทัดเดียวก็ตัดสินไม่ได้

> ⚠️ **สำคัญ:** ต้องใช้ `HDFS.log` ตัวเต็ม ห้ามใช้ `HDFS_2k.log` เพราะไฟล์ 2k เป็นบรรทัดที่สุ่มมา session จะไม่ครบ (แทบทุก session เหลือ event เดียว) ทำให้โมเดลไม่มีลำดับให้เรียนรู้

## Step 1: ตั้งค่า Drain3 + Masking

### หลักการ:
Drain3 จะจับ log ที่ "หน้าตาเหมือนกัน" มารวมเป็น template เดียว แล้วแทนส่วนที่เปลี่ยนด้วย `<*>`

แต่ถ้าปล่อยให้ Drain3 เดาเอง มันจะสับสนกับค่าที่เปลี่ยนทุกบรรทัด เช่น block id, IP, ตัวเลข ทำให้ template แตกเป็นหลายอันเกินจริง
→ เราจึงใช้ **Masking** คือแทนค่าพวกนี้ด้วยตัวแทนก่อน (`<BLK>`, `<IP>`, `<NUM>`) แล้วค่อยให้ Drain3 ทำงาน

**Analogy:** เหมือนการทำ IOC ให้เป็น pattern เวลาเขียน detection rule เราไม่สนว่า IP คืออะไร สนแค่ว่า "มี IP อยู่ตรงนี้"

### ทำไมต้องเซฟ state?
ตอนทำ Streamlit app เราต้อง map log ใหม่ให้ได้ Event ID ชุดเดียวกับตอนเทรน ถ้าไม่เซฟไว้ `E5` ตอนเทรนกับ `E5` ตอนใช้งานจริงอาจเป็นคนละ template กัน

In [1]:
import os
import re
from collections import defaultdict

import pandas as pd
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig
from drain3.masking import MaskingInstruction
from drain3.file_persistence import FilePersistence

# ===== Paths =====
LOG_PATH    = '../data/raw/HDFS/HDFS.log'                       # ไฟล์เต็ม (ไม่ใช่ HDFS_2k.log)
LABEL_PATH  = '../data/raw/HDFS/preprocessed/anomaly_label.csv'
TRACES_PATH = '../data/raw/HDFS/preprocessed/Event_traces.csv'  # เฉลยของ Loghub ใช้เช็กผล
STATE_PATH  = '../checkpoints/drain3_hdfs.bin'                  # Drain state สำหรับ Streamlit app
OUTPUT_DIR  = '../data/processed/HDFS'

# ลบ state เก่าทิ้งทุกครั้ง ไม่งั้น FilePersistence จะโหลดของเดิมมาทำต่อ (template จะปนกัน)
os.makedirs('../checkpoints', exist_ok=True)
if os.path.exists(STATE_PATH):
    os.remove(STATE_PATH)


def build_config() -> TemplateMinerConfig:
    """สร้าง config ของ Drain3 พร้อม masking สำหรับ HDFS log"""
    config = TemplateMinerConfig()
    config.masking_instructions = [
        MaskingInstruction(r'blk_-?\d+', 'BLK'),                     # block id
        MaskingInstruction(r'(\d{1,3}\.){3}\d{1,3}(:\d+)?', 'IP'),   # IP หรือ IP:port
        MaskingInstruction(r'\b\d+\b', 'NUM'),                       # ตัวเลขอื่นๆ
    ]
    config.profiling_enabled = False
    return config


# miner ตัวหลัก (ใช้จริง + เซฟ state)
template_miner = TemplateMiner(FilePersistence(STATE_PATH), build_config())
print('Drain3 พร้อมใช้งาน ✅')

Drain3 พร้อมใช้งาน ✅


### ทดลองดูผลของ Masking (Demo)
ใช้ miner **อีกตัวแยก** (`demo_miner`) เพื่อไม่ให้ข้อความทดสอบไปปนกับ miner ตัวหลัก

In [2]:
demo_miner = TemplateMiner(config=build_config())   # ไม่ใส่ persistence = ไม่เซฟ

demo_logs = [
    'Receiving block blk_123 src: /10.0.0.1:5000 dest: /10.0.0.2:50010',
    'Receiving block blk_-999 src: /192.168.1.1:4321 dest: /192.168.1.2:50010',
]
for text in demo_logs:
    r = demo_miner.add_log_message(text)
    print(f"{r['change_type']:<26} E{r['cluster_id']} → {r['template_mined']}")

# สังเกต: 2 บรรทัดได้ Event ID เดียวกัน เพราะ masking แทน block id / IP ให้แล้ว

cluster_created            E1 → Receiving block <BLK> src: /<IP> dest: /<IP>
none                       E1 → Receiving block <BLK> src: /<IP> dest: /<IP>


## Step 2: Parse ไฟล์เต็ม + สร้าง Session

### หลักการ:
อ่านทีละบรรทัด → แยก `content` ด้วย regex ตัวเดียวกับใน EDA → ส่งให้ Drain3 ได้ Event ID → เอาไปต่อท้าย session ของ `block_id` นั้น

**จุดที่แก้จากเวอร์ชันก่อน:**
- ใช้ `findall` แทน `search` เพราะบางบรรทัดมีหลาย block (เช่น `ask ... to delete blk_A blk_B ...`) ต้องใส่ event ให้ **ทุก** block
- ใช้ `eid_cache` เก็บ string `"E5"` ตัวเดียวแล้วใช้ซ้ำ เพราะมี 11 ล้านบรรทัด ถ้าสร้าง string ใหม่ทุกครั้งจะกิน RAM มาก

⏱️ **ใช้เวลาประมาณ 10–30 นาที** (ขึ้นกับเครื่อง)

In [3]:
# regex เดียวกับใน 01_eda: <Date> <Time> <Pid> <Level> <Component>: <Content>
hdfs_pattern = re.compile(r'\d{6}\s+\d{6}\s+\d+\s+\w+\s+\S+:\s+(?P<content>.+)')
block_regex = re.compile(r'blk_-?\d+')

sessions: dict[str, list[str]] = defaultdict(list)   # block_id → [E5, E22, E5, ...]
eid_cache: dict[int, str] = {}                        # cluster_id → "E{id}" (ใช้ซ้ำ ประหยัด RAM)
total_lines = 0
unparsed_lines = 0

with open(LOG_PATH, 'r') as f:
    for total_lines, line in enumerate(f, 1):
        m = hdfs_pattern.match(line.strip())
        if not m:
            unparsed_lines += 1
            continue
        content = m.group('content')

        cluster_id = template_miner.add_log_message(content)['cluster_id']
        event_id = eid_cache.setdefault(cluster_id, f'E{cluster_id}')

        # set() กันกรณี block เดิมโผล่ซ้ำในบรรทัดเดียว
        for blk in set(block_regex.findall(content)):
            sessions[blk].append(event_id)

        if total_lines % 1_000_000 == 0:
            print(f'Processed {total_lines:,} lines...')

template_miner.save_state('parsing done')

print('\n=== Summary ===')
print(f'Total lines:     {total_lines:,}')
print(f'Unparsed lines:  {unparsed_lines:,}')
print(f'Sessions:        {len(sessions):,}   (ควรได้ประมาณ 575,061)')
print(f'Event templates: {len(template_miner.drain.clusters)}   (Loghub มี 29 — ถ้าเกิน 50 ต้องปรับ masking)')

Processed 1,000,000 lines...
Processed 2,000,000 lines...
Processed 3,000,000 lines...
Processed 4,000,000 lines...
Processed 5,000,000 lines...
Processed 6,000,000 lines...
Processed 7,000,000 lines...
Processed 8,000,000 lines...
Processed 9,000,000 lines...
Processed 10,000,000 lines...
Processed 11,000,000 lines...

=== Summary ===
Total lines:     11,175,629
Unparsed lines:  0
Sessions:        575,061   (ควรได้ประมาณ 575,061)
Event templates: 45   (Loghub มี 29 — ถ้าเกิน 50 ต้องปรับ masking)


### ดู Template ทั้งหมดที่ Drain3 หาได้
เรียงตามจำนวนครั้งที่เจอ เพื่อเช็กด้วยตาว่า template สมเหตุสมผล ไม่มี block id หรือ IP หลุดมา

In [4]:
templates_df = pd.DataFrame(
    [{'EventId': f'E{c.cluster_id}', 'Count': c.size, 'Template': c.get_template()}
     for c in template_miner.drain.clusters]
).sort_values('Count', ascending=False).reset_index(drop=True)

pd.set_option('display.max_colwidth', 120)
templates_df

,EventId,Count,Template
0,E1,1723232,Receiving block <BLK> src: /<IP> dest: /<IP>
1,E5,1719741,BLOCK* NameSystem.addStoredBlock: blockMap updated: <IP> is added to <BLK> size <NUM>
2,E3,1706728,PacketResponder <NUM> for block <BLK> <*>
3,E4,1706514,Received block <BLK> of size <NUM> from /<IP>
4,E17,1402047,Deleting block <BLK> file <*>
5,E32,1396174,BLOCK* NameSystem.delete: <BLK> is added to invalidSet of <IP>
6,E2,575061,BLOCK* NameSystem.allocateBlock: <*> <BLK>
7,E10,428726,<IP> Served block <BLK> to /<IP>
8,E35,356207,<IP>:Got exception while serving <BLK> to /<IP>:
9,E13,120036,Verification succeeded for <BLK>


## Step 3: Sanity Check — เทียบกับเฉลยของ Loghub

### หลักการ:
Loghub มีไฟล์ `Event_traces.csv` ที่ parse ไว้ให้แล้ว เราเอาความยาว session ของเรามาเทียบกับของเขา
ถ้าตรงกันเกือบทั้งหมด = เราแบ่ง session ถูกแล้ว ✅

(Event ID ของเรากับของ Loghub จะเป็นคนละเลขกัน ไม่เป็นไร เพราะ Drain3 ตั้งเลขตามลำดับที่เจอ เราเทียบแค่ **ความยาว**)

In [5]:
traces = pd.read_csv(TRACES_PATH, usecols=['BlockId', 'Features'])
traces['ref_len'] = traces['Features'].str.strip('[]').str.split(',').str.len()
ours = pd.Series({blk: len(seq) for blk, seq in sessions.items()}, name='our_len')

cmp = traces.set_index('BlockId').join(ours)
match_pct = (cmp['ref_len'] == cmp['our_len']).mean() * 100

print(f'Session ที่ความยาวตรงกับ Loghub: {match_pct:.2f}%   (ควร ≥ 95%)')
print(f'Session ใน Loghub ที่เราหาไม่เจอ: {cmp["our_len"].isna().sum():,}')
print(f'\nความยาว session ของเรา: median = {ours.median():.0f}, mean = {ours.mean():.1f}   (EDA: median 22, mean 21.9)')

print('\n=== ตัวอย่าง 3 session แรก ===')
for i, (blk, seq) in enumerate(sessions.items()):
    if i >= 3:
        break
    print(f'Session {i+1} [{blk}] ยาว {len(seq)} events:')
    print(f'   {seq}')

Session ที่ความยาวตรงกับ Loghub: 100.00%   (ควร ≥ 95%)
Session ใน Loghub ที่เราหาไม่เจอ: 0

ความยาว session ของเรา: median = 19, mean = 19.4   (EDA: median 22, mean 21.9)

=== ตัวอย่าง 3 session แรก ===
Session 1 [blk_-1608999687919862906] ยาว 269 events:
   ['E1', 'E2', 'E1', 'E1', 'E3', 'E3', 'E4', 'E4', 'E3', 'E4', 'E5', 'E5', 'E5', 'E6', 'E1', 'E7', 'E6', 'E1', 'E8', 'E9', 'E5', 'E5', 'E10', 'E9', 'E6', 'E6', 'E1', 'E1', 'E7', 'E8', 'E5', 'E5', 'E1', 'E6', 'E1', 'E7', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E8', 'E9', 'E6', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E5', 'E5', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E10', 'E1

## Step 4: ผูก Label (Normal = 0, Anomaly = 1)

### หลักการ:
`anomaly_label.csv` บอกว่า block ไหน Normal / Anomaly เราเอามา join กับ session ด้วย `BlockId` (เหมือน INNER JOIN ใน SQL)

In [6]:
df_labels = pd.read_csv(LABEL_PATH)
df_labels['Label'] = df_labels['Label'].map({'Normal': 0, 'Anomaly': 1})

df_sessions = pd.DataFrame({
    'BlockId': list(sessions.keys()),
    'Sequence': list(sessions.values()),
})

df_dataset = pd.merge(df_sessions, df_labels, on='BlockId', how='inner')

print(f'ได้ข้อมูลพร้อมใช้งานทั้งหมด: {len(df_dataset):,} sessions\n')
print('=== ตัวอย่าง 5 แถวแรก ===')
print(df_dataset.head())

print('\n=== จำนวน Normal / Anomaly ===')
counts = df_dataset['Label'].value_counts().sort_index()
for label, n in counts.items():
    name = 'Normal (0)' if label == 0 else 'Anomaly (1)'
    print(f'{name:12s}: {n:>8,} ({n / len(df_dataset) * 100:.2f}%)')

ได้ข้อมูลพร้อมใช้งานทั้งหมด: 575,061 sessions

=== ตัวอย่าง 5 แถวแรก ===
                    BlockId  \
0  blk_-1608999687919862906   
1   blk_7503483334202473044   
2  blk_-3544583377289625738   
3  blk_-9073992586687739851   
4   blk_7854771516489510256   

                                                                                                                  Sequence  \
0  [E1, E2, E1, E1, E3, E3, E4, E4, E3, E4, E5, E5, E5, E6, E1, E7, E6, E1, E8, E9, E5, E5, E10, E9, E6, E6, E1, E1, E7...   
1                        [E1, E1, E2, E1, E3, E4, E3, E4, E3, E4, E5, E5, E5, E10, E13, E13, E32, E32, E32, E17, E17, E17]   
2  [E1, E2, E1, E1, E3, E4, E3, E4, E3, E4, E10, E5, E5, E5, E10, E10, E10, E10, E10, E10, E10, E10, E10, E10, E10, E10...   
3                        [E1, E2, E1, E1, E3, E4, E3, E4, E3, E4, E5, E5, E5, E13, E13, E13, E32, E32, E32, E17, E17, E17]   
4  [E1, E1, E2, E1, E3, E4, E3, E4, E3, E4, E5, E5, E5, E13, E13, E13, E35, E35, E35, E35, E35, E35, E35, E35,

## Step 5: แบ่ง Train / Validation / Test (70 / 15 / 15)

### หลักการ:
- **Train** → ให้โมเดลเรียน
- **Validation** → ใช้ปรับ hyperparameter / threshold (เหมือนสอบย่อย)
- **Test** → ใช้วัดผลจริงครั้งสุดท้าย ห้ามแตะระหว่างพัฒนา (เหมือนสอบไฟนอล)

ใช้ `stratify` เพื่อให้สัดส่วน Anomaly (~2.9%) เท่ากันทุกชุด และ `random_state=42` ให้ได้ผลเหมือนเดิมทุกครั้งที่รัน

In [7]:
from sklearn.model_selection import train_test_split

# เก็บ sequence เป็น string คั่นด้วยช่องว่าง เพื่อบันทึกลง CSV ได้ง่าย
df_export = df_dataset.copy()
df_export['Sequence'] = df_export['Sequence'].apply(' '.join)

train_df, temp_df = train_test_split(
    df_export, test_size=0.30, random_state=42, stratify=df_export['Label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42, stratify=temp_df['Label']
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
train_df.to_csv(f'{OUTPUT_DIR}/train.csv', index=False)
val_df.to_csv(f'{OUTPUT_DIR}/val.csv', index=False)
test_df.to_csv(f'{OUTPUT_DIR}/test.csv', index=False)

print(f'บันทึกไฟล์เรียบร้อยที่: {OUTPUT_DIR}/\n')
for name, d in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f'{name:5s}: {len(d):>8,} sessions | Anomaly {d["Label"].sum():>6,} ({d["Label"].mean() * 100:.2f}%)')

บันทึกไฟล์เรียบร้อยที่: ../data/processed/HDFS/

Train:  402,542 sessions | Anomaly 11,787 (2.93%)
Val  :   86,259 sessions | Anomaly  2,525 (2.93%)
Test :   86,260 sessions | Anomaly  2,526 (2.93%)


## 📝 สรุป

1. Parse `HDFS.log` ตัวเต็มด้วย Drain3 + masking → ได้ event template ใกล้เคียงกับของ Loghub
2. จัดกลุ่มเป็น session ตาม `block_id` (ใช้ `findall` รองรับบรรทัดที่มีหลาย block) และเช็กความยาวกับ `Event_traces.csv` แล้ว
3. เซฟ Drain state ไว้ที่ `checkpoints/drain3_hdfs.bin` สำหรับ Streamlit app (ตอนใช้งานจริงให้ใช้ `template_miner.match(content)` แทน `add_log_message` เพื่อไม่ให้สร้าง template ใหม่)
4. แบ่งข้อมูล 70 / 15 / 15 แบบ stratified → `data/processed/HDFS/`

**ต่อไป:** `03_baseline_models.ipynb`